# Kaggle: serve and verify a quantized checkpoint

This is the second half of the end-to-end test. Attach the Kaggle Dataset created by `kaggle_quantize_tinyllama.ipynb` as an input, then set **Accelerator → GPU T4 x2** and turn **Internet** on.

This notebook deliberately starts with a new Kaggle runtime. vLLM can install a different Torch build than `llmcompressor`; testing the saved artifact in a clean serving environment is both more reliable and the behavior the project needs to support.

In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator, then restart the session.'
capability = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)} (compute capability {capability[0]}.{capability[1]})')
assert capability >= (7, 5), 'Choose T4 x2 rather than P100; vLLM requires compute capability 7.5+.'

WORKDIR = Path('/kaggle/working')
REPOSITORY = 'https://github.com/lelouch0204/quantization-profiler.git'
BRANCH = 'kaggle-vllm-test-drive'
CHECKOUT = WORKDIR / 'quantization-profiler'


In [ ]:
if not CHECKOUT.exists():
    !git clone --branch {BRANCH} {REPOSITORY} {CHECKOUT}
else:
    !git -C {CHECKOUT} fetch origin {BRANCH}
    !git -C {CHECKOUT} checkout {BRANCH}
    !git -C {CHECKOUT} pull --ff-only origin {BRANCH}

# This is intentionally a clean serving environment.
%pip install -q vllm requests
print('Restart the Kaggle session now, then continue at the next cell.')


## Restart checkpoint

Use Kaggle's **Restart Session** action after the installation cell. Do not rerun the quantization notebook or install `llmcompressor` in this runtime. Once the kernel is back, continue below.

In [ ]:
import sys

sys.path.insert(0, str(CHECKOUT / 'src'))
from vllm_server_manager import VLLMServerManager

candidates = [
    config.parent
    for config in Path('/kaggle/input').rglob('config.json')
    if (config.parent / 'quant_key.json').is_file()
]
assert candidates, (
    'No quantized checkpoint was found. Attach the Dataset produced by the quantization notebook, then restart.'
)
assert len(candidates) == 1, f'Expected one checkpoint input, found: {candidates}'
checkpoint = candidates[0]
print(f'Serving: {checkpoint}')
print((checkpoint / 'quant_key.json').read_text())


In [ ]:
manager = VLLMServerManager()
manager.start_server(str(checkpoint), port=8000)
manager.wait_for_health(timeout=600)
print('vLLM health check passed')


In [ ]:
import requests

models = requests.get('http://localhost:8000/v1/models', timeout=30)
models.raise_for_status()
model_id = models.json()['data'][0]['id']

response = requests.post(
    'http://localhost:8000/v1/completions',
    json={
        'model': model_id,
        'prompt': 'The capital of France is',
        'max_tokens': 12,
        'temperature': 0,
    },
    timeout=120,
)
response.raise_for_status()
completion = response.json()['choices'][0]['text']
assert completion.strip(), 'vLLM returned an empty completion'
print(completion)


In [ ]:
manager.terminate()
print('vLLM stopped and GPU memory released')


A successful health check and non-empty completion verifies the intended contract: `Quantizer` emitted a compressed-tensors checkpoint that vLLM can discover and serve without a manually supplied quantization flag.